# ADMET Results Scoring

This notebook analyzes the ADMET predictions and scores candidates based on a specific profile for **Septic Shock (IV administration)**.
We apply hard filters for toxicity and calculate a weighted score based on key properties.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load Data

Load results from `ligands_admet.csv`.

In [2]:
ligands_input_path = Path("..") / "data" / "processed" / "ligands_admet.csv"
ligands_output_path = Path("..") / "data" / "processed" / "best_sepsis_candidates.csv"
head_size = 20

df = pd.read_csv(ligands_input_path)
print(df.head())

## Scoring Strategy

We define a custom weighted scoring system designed for IV Sepsis treatment.

### 1. Hard Filters
Remove molecules with high toxicity risks immediately:
- **hERG (Cardiotoxicity)**: Must be < 0.5 (lower is better/safer).
- **AMES (Mutagenicity)**: Must be < 0.7 (lower is better/safer).

### 2. Weights
Different properties contribute differently to the final score:
- **hERG (4.0)**: Critical safety concern.
- **Solubility (3.5)**: Essential for IV drugs.
- **DILI (3.0)**: Drug-Induced Liver Injury risk.
- **AMES (2.0)**: Mutagenicity risk.
- **QED (2.0)**: General drug-likeness.
- **BBB (1.0)**: Blood-Brain Barrier penetration (less critical for peripheral sepsis but considered).

### 3. Normalization
All scores are normalized to a 0-100 scale:
- 'lower_better' metrics (Toxicity) are inverted: `100 * (1 - value)`.
- 'higher_better' metrics (QED) are scaled directly: `100 * value`.
- Solubility (logS) is normalized from a typical range of -6 (poor) to -2 (good).

In [3]:
def scale_value(val, type='lower_better'):
    val = np.clip(val, 0, 1)
    if type == 'lower_better':
        return 1.0 - val
    return val

def normalize_log_s(val):
    # -2 (Unknown/Good), -6 (Poor) -> scale to 0..1
    min_val, max_val = -6, -2
    norm = (val - min_val) / (max_val - min_val)
    return np.clip(norm, 0, 1)

print(f"--- Mode: Septic Shock (IV administration) ---")

# 1. HARD FILTERS
if 'hERG' in df.columns:
    df = df[df['hERG'] < 0.5].copy()
if 'AMES' in df.columns:
    df = df[df['AMES'] < 0.7].copy()

# 2. WEIGHTS
weights = {
    'hERG': 4.0,
    'AMES': 2.0,
    'DILI': 3.0,
    'Solubility': 3.5,
    'HIA_Hou': 0.0,
    'QED': 2.0,
    'BBB': 1.0
}

# 3. CALCULATE CATEGORY SCORES (0-100)
df['score_hERG'] = df['hERG'].apply(lambda x: scale_value(x, 'lower_better')) * 100
df['score_AMES'] = df['AMES'].apply(lambda x: scale_value(x, 'lower_better')) * 100
df['score_DILI'] = df['DILI'].apply(lambda x: scale_value(x, 'lower_better')) * 100
df['score_Sol']  = df['Solubility_AqSolDB'].apply(normalize_log_s) * 100
df['score_QED']  = df['QED'].apply(lambda x: scale_value(x, 'higher_better')) * 100

# BBB Logic
if 'BBB_Martins' in df.columns:
    df['score_BBB'] = df['BBB_Martins'].apply(lambda x: scale_value(x, 'lower_better')) * 100
else:
    df['score_BBB'] = 0

# 4. COMPUTE FINAL WEIGHTED SCORE
weighted_sum = (
    df['score_hERG'] * weights['hERG'] +
    df['score_AMES'] * weights['AMES'] +
    df['score_DILI'] * weights['DILI'] +
    df['score_Sol']  * weights['Solubility'] +
    df['score_QED']  * weights['QED']
)

total_weight = (
    weights['hERG'] + 
    weights['AMES'] + 
    weights['DILI'] + 
    weights['Solubility'] + 
    weights['QED']
)

if 'BBB_Martins' in df.columns:
    weighted_sum += df['score_BBB'] * weights['BBB']
    total_weight += weights['BBB']

df['Sepsis_Score_Norm'] = weighted_sum / total_weight

# 5. FORMAT AND SAVE
df_sorted = df.sort_values(by='Sepsis_Score_Norm', ascending=False)

score_cols = ['score_hERG', 'score_AMES', 'score_DILI', 'score_Sol', 'score_QED']
if 'score_BBB' in df.columns:
    score_cols.append('score_BBB')

df_sorted[score_cols + ['Sepsis_Score_Norm']] = df_sorted[score_cols + ['Sepsis_Score_Norm']].round(1)

raw_cols = ['hERG', 'Solubility_AqSolDB', 'DILI', 'AMES', 'QED', 'BBB_Martins']
raw_cols_present = [c for c in raw_cols if c in df_sorted.columns]
df_sorted[raw_cols_present] = df_sorted[raw_cols_present].round(3)

final_columns_to_save = [
    'COMB', 'SMILES', 'Sepsis_Score_Norm',
    'score_hERG', 'hERG',
    'score_Sol',  'Solubility_AqSolDB',
    'score_DILI', 'DILI',
    'score_AMES', 'AMES',
    'score_QED',  'QED'
]
if 'score_BBB' in df_sorted.columns:
    final_columns_to_save.extend(['score_BBB', 'BBB_Martins'])

final_columns_to_save = [c for c in final_columns_to_save if c in df_sorted.columns]

print(f"\nTop 3 Candidates (Scores 0-100):")
print(df_sorted[['Sepsis_Score_Norm', 'score_hERG', 'score_Sol', 'score_DILI']].head(3).to_string(index=False))

df_sorted.head(head_size)[final_columns_to_save].to_csv(ligands_output_path, index=False)